In [ ]:
import pickle

import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt
import torch

from tools.geometry import generate_detector

print("Imports successful")

In [ ]:
# --------------------------
# Extract convergence histories
# --------------------------
def extract_histories(all_event_results):
    """
    Extract convergence histories for all events.
    
    Returns:
        Dictionary with arrays for each metric, shape (n_events, n_iterations)
    """
    histories = {
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
        'combined_losses': [],
        'vertex_losses': [],
        'counts_losses': [],
        'energy_losses': []
    }
    
    for event_result in all_event_results:
        opt_results = event_result['optimization_results']
        history = opt_results['history']
        
        histories['position_errors'].append(history['position_errors'])
        histories['direction_errors'].append(history['direction_errors'])
        histories['t0_errors'].append(history['t0_errors'])
        histories['energy_errors'].append(history['energy_errors'])
        histories['combined_losses'].append(history['combined_losses'])
        histories['vertex_losses'].append(history['vertex_losses'])
        histories['counts_losses'].append(history['counts_losses'])
        histories['energy_losses'].append(history['energy_losses'])
    
    # Convert to numpy arrays
    for key in histories:
        histories[key] = np.array(histories[key])
    
    return histories

# --------------------------
# Compute statistics
# --------------------------
def compute_statistics(data_array):
    """
    Compute mean, median, 68th percentile, and 90th percentile.
    
    Args:
        data_array: numpy array of shape (n_events, n_iterations)
    
    Returns:
        Dictionary with statistical measures
    """
    return {
        'mean': np.mean(data_array, axis=0),
        'median': np.median(data_array, axis=0),
        'percentile_68': np.percentile(data_array, 68, axis=0),
        'percentile_90': np.percentile(data_array, 90, axis=0)
    }

In [ ]:
import pickle
import numpy as np


common_path = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/'
results_files = [
    common_path + 'results_5k_nrays.pkl',
    common_path + 'results_10k_nrays.pkl',
    common_path + 'results_25k_nrays.pkl',
    common_path + 'results_50k_nrays.pkl',
    common_path + 'results_100k_nrays.pkl',
    common_path + 'results_150k_nrays.pkl',
]


list_of_x_variable = []
list_of_histories = []
list_of_stats = []
results_summaries = []
for i, results_file in enumerate(results_files):
    print(results_file)
    with open(results_file, 'rb') as f:
        results_summary = pickle.load(f)
        results_summaries.append(results_summary)

    print(f"Loaded results from: {results_file}")
    print(f"Nphot: {results_summaries[-1]['config']['nphot']}")
    list_of_x_variable.append(results_summaries[-1]['config']['nphot'])

    all_event_results = results_summary['all_event_results']
    histories = extract_histories(all_event_results)

    # Compute statistics for all metrics
    stats = {}
    for key in histories:
        stats[key] = compute_statistics(histories[key])

    n_events, n_iterations = histories['position_errors'].shape

    # --------------------------
    # Convert energy errors → momentum errors (%)
    # --------------------------
    # Constants
    m_mu = 105.658  # GeV (muon mass)
    T_mu = 1050.0    # GeV (kinetic energy)
    E_total = T_mu + m_mu
    p_mu = np.sqrt(E_total**2 - m_mu**2)
    conversion_factor = E_total / p_mu
    momentum_errors_percent = (conversion_factor * (histories['energy_errors'] / E_total)) * 100
    histories['momentum_errors_percent'] = momentum_errors_percent
    stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)

    list_of_histories.append(histories)
    list_of_stats.append(stats)


In [ ]:
common_path = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/'
results_files = [
    common_path + 'speed_results_5k_nrays.pkl',
    common_path + 'speed_results_10k_nrays.pkl',
    common_path + 'speed_results_25k_nrays.pkl',
    common_path + 'speed_results_50k_nrays.pkl',
    common_path + 'speed_results_100k_nrays.pkl',
    common_path + 'speed_results_150k_nrays.pkl'
]

list_of_x_variable = []
list_of_histories = []
list_of_stats = []
results_summaries = []
for i, results_file in enumerate(results_files):
    print(results_file)
    with open(results_file, 'rb') as f:
        results_summary = pickle.load(f)
        results_summaries.append(results_summary)

    print(f"Loaded results from: {results_file}")
    print(f"Nphot: {results_summaries[-1]['config']['nphot']}")
    list_of_x_variable.append(results_summaries[-1]['config']['nphot'])

    all_event_results = results_summary['all_event_results']
    histories = extract_histories(all_event_results)

    # Compute statistics for all metrics
    stats = {}
    for key in histories:
        stats[key] = compute_statistics(histories[key])

    n_events, n_iterations = histories['position_errors'].shape

    # --------------------------
    # Convert energy errors → momentum errors (%)
    # --------------------------
    # Constants
    m_mu = 0.105658  # GeV (muon mass)
    T_mu = 1050.0    # GeV (kinetic energy)
    E_total = T_mu + m_mu
    p_mu = np.sqrt(E_total**2 - m_mu**2)
    conversion_factor = E_total / p_mu
    momentum_errors_percent = (conversion_factor * (histories['energy_errors'] / E_total)) * 100
    histories['momentum_errors_percent'] = momentum_errors_percent
    stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)

    list_of_histories.append(histories)
    list_of_stats.append(stats)

total_times_mean, total_times_std = [], []
adam_times_mean, adam_times_std = [], []

# time numbers are calculated in a single job to avoid hardware-driven variations... (5,10,25,50,100,150)k Nrays
for results_summary in results_summaries:
    all_event_results = results_summary["all_event_results"]
    total_t = np.array([ev["total_event_time"] for ev in all_event_results])
    adam_t = np.array([ev.get("adam_optimization_time", np.nan) for ev in all_event_results])
    total_times_mean.append(np.nanmean(total_t))
    total_times_std.append(np.nanstd(total_t))
    adam_times_mean.append(np.nanmean(adam_t))
    adam_times_std.append(np.nanstd(adam_t))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

ci_level_val = 90
line_color = "navy"

def bootstrap_percentile_ci(data, percentile=95, n_bootstrap=1000, ci_level=ci_level_val):
    bootstrap_estimates = []
    n = len(data)
    for _ in range(n_bootstrap):
        bootstrap_sample = np.random.choice(data, size=n, replace=True)
        bootstrap_estimates.append(np.percentile(bootstrap_sample, percentile))
    bootstrap_estimates = np.array(bootstrap_estimates)
    alpha = (100 - ci_level) / 2
    ci_lower = np.percentile(bootstrap_estimates, alpha)
    ci_upper = np.percentile(bootstrap_estimates, 100 - alpha)
    return {
        "estimate": np.percentile(data, percentile),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "std_error": np.std(bootstrap_estimates),
    }

for results_summary in results_summaries:
    all_event_results = results_summary["all_event_results"]
    total_t = np.array([ev["total_event_time"] for ev in all_event_results])
    adam_t = np.array([ev.get("adam_optimization_time", np.nan) for ev in all_event_results])


# ==========================================================
# Bootstrap metrics
# ==========================================================
n_iter = -1
np.random.seed(42)

metrics = {
    "position_errors": {"label": "Pos. error (cm)", "scale": 100},
    "direction_errors": {"label": "Dir. error (°)", "scale": 1},
    "t0_errors": {"label": "t₀ error (ns)", "scale": 1},
    "momentum_errors_percent": {"label": "Mom. error (%)", "scale": 1},
}

valid_histories = [h for h in list_of_histories if isinstance(h.get("position_errors", None), np.ndarray)]
metric_results = {key: {"y": [], "yerr": []} for key in metrics}

for histories in valid_histories:
    for key, meta in metrics.items():
        data = histories[key][:, n_iter] * meta["scale"]
        boot = bootstrap_percentile_ci(data, percentile=68)
        metric_results[key]["y"].append(boot["estimate"])
        yerr = 0.5 * ((boot["ci_upper"] - boot["estimate"]) + (boot["estimate"] - boot["ci_lower"]))
        metric_results[key]["yerr"].append(yerr)

# ==========================================================
# Fit function
# ==========================================================
def exp_like(x, a, b, c):
    """y = a * b^x + c"""
    return a * (b ** x) + c

# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(5, 1, figsize=(6, 10), sharex=True)
plt.subplots_adjust(hspace=0.05)

xdata = np.array(list_of_x_variable) / 1000.0  # <--- rescale x by 1000
xlabels = [f"{int(x)}" for x in xdata]

for ax, (key, meta) in zip(axes[:4], metrics.items()):
    y = np.array(metric_results[key]["y"])
    yerr = np.array(metric_results[key]["yerr"])

    ax.errorbar(
        xdata, y, yerr=yerr, fmt="o", capsize=4, color=line_color, label=meta["label"]
    )

    # Fit with error weighting
    try:
        popt, _ = curve_fit(
            exp_like,
            xdata,
            y,
            sigma=yerr,
            absolute_sigma=True,
            p0=[y.max() - y.min(), 0.99, y.min()],
            maxfev=10000,
        )
        xfit = np.linspace(xdata.min(), xdata.max(), 200)
        yfit = exp_like(xfit, *popt)
        ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)
        print(y)

        # Annotate fit parameters
        a, b, c = popt
        ax.text(
            0.55, 0.9,
            f"$y = {a:.2f}\\times\\,{b:.3f}^x + {c:.2f}$",
            transform=ax.transAxes, fontsize=10, color="cornflowerblue",
            ha="left", va="top",
        )
    except RuntimeError:
        print(f"⚠️ Fit failed for {key}")

    # Set fixed Y-axis limits for each metric
    if key == "position_errors":
        ax.set_ylim(10, 50)
    elif key == "direction_errors":
        ax.set_ylim(0.7, 2.3)
    elif key == "t0_errors":
        ax.set_ylim(0, 2.1)
    elif key == "momentum_errors_percent":
        ax.set_ylim(0, 16)

    ax.set_ylabel(meta["label"])
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

# ==========================================================
# Bottom panel — total & Adam optimization times
# ==========================================================
ax_time = axes[-1]
ax_time.errorbar(
    xdata, total_times_mean, yerr=total_times_std,
    fmt="o-", capsize=4, color="tab:red", label="Total"
)
ax_time.errorbar(
    xdata, adam_times_mean, yerr=adam_times_std,
    fmt="s--", capsize=4, color="tab:orange", label="Adam"
)

ax_time.set_ylabel("Time per event (s)")
ax_time.set_xlabel("Number of Rays ($\\times10^3$)")
ax_time.grid(True, alpha=0.3)
ax_time.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
ax_time.legend(frameon=False, loc="lower right")
ax_time.set_ylim(0)
ax_time.set_xlim(0)

axes[-1].set_xticks(xdata)
axes[-1].set_xticklabels(xlabels)

fig.align_ylabels(axes)
plt.savefig('figures/tracking_performance_vs_nrays.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import pickle
import numpy as np


common_path = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/'
results_files = [
    common_path + 'results_2k_geom_opt.pkl',
    common_path + 'results_3k_geom_opt.pkl',
    common_path + 'results_4k_geom_opt.pkl',
    common_path + 'results_5k_geom_opt.pkl',
    common_path + 'results_6k_geom_opt.pkl',
    common_path + 'results_7k_geom_opt.pkl',
    common_path + 'results_8k_geom_opt.pkl',
    common_path + 'results_9k_geom_opt.pkl',
    common_path + 'results_10k_geom_opt.pkl',
    common_path + 'results_11k_geom_opt.pkl',
    common_path + 'results_12k_geom_opt.pkl',
    common_path + 'results_13k_geom_opt.pkl',
    common_path + 'results_14k_geom_opt.pkl',
    common_path + 'results_15k_geom_opt.pkl',
    common_path + 'results_16k_geom_opt.pkl',
    common_path + 'results_17k_geom_opt.pkl',
    common_path + 'results_18k_geom_opt.pkl',
    common_path + 'results_19k_geom_opt.pkl',
]

list_of_x_variable = []
list_of_histories = []
list_of_stats = []
results_summaries = []
for i, results_file in enumerate(results_files):
    with open(results_file, 'rb') as f:
        results_summary = pickle.load(f)
        results_summaries.append(results_summary)

    print(f"Loaded results from: {results_file}")
    detector_config_filename = results_summaries[-1]['config']['detector_file']
    detector = generate_detector(detector_config_filename)
    list_of_x_variable.append(len(detector.all_points))

    all_event_results = results_summary['all_event_results']
    histories = extract_histories(all_event_results)

    # Compute statistics for all metrics
    stats = {}
    for key in histories:
        stats[key] = compute_statistics(histories[key])

    n_events, n_iterations = histories['position_errors'].shape

    # --------------------------
    # Convert energy errors → momentum errors (%)
    # --------------------------
    # Constants
    m_mu = 105.658  # GeV (muon mass)
    T_mu = 1050.0    # GeV (kinetic energy)
    E_total = T_mu + m_mu
    p_mu = np.sqrt(E_total**2 - m_mu**2)
    conversion_factor = E_total / p_mu
    momentum_errors_percent = (conversion_factor * (histories['energy_errors'] / E_total)) * 100
    histories['momentum_errors_percent'] = momentum_errors_percent
    stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)

    list_of_histories.append(histories)
    list_of_stats.append(stats)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

ci_level_val = 68
line_color = "navy"

def bootstrap_percentile_ci(data, percentile=95, n_bootstrap=1000, ci_level=ci_level_val):
    bootstrap_estimates = []
    n = len(data)
    for _ in range(n_bootstrap):
        bootstrap_sample = np.random.choice(data, size=n, replace=True)
        bootstrap_estimates.append(np.percentile(bootstrap_sample, percentile))
    bootstrap_estimates = np.array(bootstrap_estimates)
    alpha = (100 - ci_level) / 2
    ci_lower = np.percentile(bootstrap_estimates, alpha)
    ci_upper = np.percentile(bootstrap_estimates, 100 - alpha)
    return {
        "estimate": np.percentile(data, percentile),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "std_error": np.std(bootstrap_estimates),
    }

# ==========================================================
# Timing metrics
# ==========================================================
total_times_mean, total_times_std = [], []
adam_times_mean, adam_times_std = [], []

for results_summary in results_summaries:
    all_event_results = results_summary["all_event_results"]
    total_t = np.array([ev["total_event_time"] for ev in all_event_results])
    adam_t = np.array([ev.get("adam_optimization_time", np.nan) for ev in all_event_results])
    total_times_mean.append(np.nanmean(total_t))
    total_times_std.append(np.nanstd(total_t))
    adam_times_mean.append(np.nanmean(adam_t))
    adam_times_std.append(np.nanstd(adam_t))

# ==========================================================
# Bootstrap metrics
# ==========================================================
n_iter = -1
np.random.seed(42)

metrics = {
    "position_errors": {"label": "Pos. error (cm)", "scale": 100},
    "direction_errors": {"label": "Dir. error (°)", "scale": 1},
    "t0_errors": {"label": "t₀ error (ns)", "scale": 1},
    "momentum_errors_percent": {"label": "Mom. error (%)", "scale": 1},
}

valid_histories = [h for h in list_of_histories if isinstance(h.get("position_errors", None), np.ndarray)]
metric_results = {key: {"y": [], "yerr": []} for key in metrics}

for histories in valid_histories:
    for key, meta in metrics.items():
        data = histories[key][:, n_iter] * meta["scale"]
        boot = bootstrap_percentile_ci(data, percentile=68)
        metric_results[key]["y"].append(boot["estimate"])
        yerr = 0.5 * ((boot["ci_upper"] - boot["estimate"]) + (boot["estimate"] - boot["ci_lower"]))
        metric_results[key]["yerr"].append(yerr)

# ==========================================================
# Fit function
# ==========================================================
def exp_like(x, a, b, c):
    """y = a * b^x + c"""
    return a * (b ** x) + c

# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(4, 1, figsize=(6, 10), sharex=True)
plt.subplots_adjust(hspace=0.05)

xdata = np.array(list_of_x_variable) / 1000.0

for ax, (key, meta) in zip(axes[:4], metrics.items()):
    y = np.array(metric_results[key]["y"])
    yerr = np.array(metric_results[key]["yerr"])

    ax.errorbar(
        xdata, y, yerr=yerr, fmt="o", capsize=4, color=line_color, label=meta["label"]
    )

    # Fit with error weighting
    try:
        popt, _ = curve_fit(
            exp_like,
            xdata,
            y,
            sigma=yerr,
            absolute_sigma=True,
            p0=[y.max() - y.min(), 0.99, y.min()],
            maxfev=10000,
        )
        xfit = np.linspace(xdata.min(), xdata.max(), 200)
        yfit = exp_like(xfit, *popt)
        ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)

        # Annotate fit parameters
        a, b, c = popt
        ax.text(
            0.55, 0.9,
            f"$y = {a:.2f}\\times\\,{b:.3f}^x + {c:.2f}$",
            transform=ax.transAxes, fontsize=10, color="cornflowerblue",
            ha="left", va="top",
        )
    except RuntimeError:
        print(f"⚠️ Fit failed for {key}")

    ax.set_ylabel(meta["label"])
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

    # Set fixed Y-axis limits for each metric
    if key == "position_errors":
        ax.set_ylim(10, 40)
    elif key == "direction_errors":
        ax.set_ylim(0.4, 1.8)
    elif key == "t0_errors":
        ax.set_ylim(0, 1.2)
    elif key == "momentum_errors_percent":
        ax.set_ylim(0, 4.5)


axes[-1].set_xlabel("Number of Sensors ($\\times10^3$)")
fig.align_ylabels(axes)
plt.savefig('figures/detector_perf_vs_num_sensors.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import pickle
import numpy as np


common_path = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/'
results_files = [
    common_path + 'energy_2.pkl',
    common_path + 'energy_3.pkl',
    common_path + 'energy_4.pkl',
    common_path + 'energy_5.pkl',
    common_path + 'energy_6.pkl',
    common_path + 'energy_7.pkl',
    common_path + 'energy_8.pkl',
    common_path + 'energy_9.pkl',
    common_path + 'energy_10.pkl',
    common_path + 'energy_11.pkl',
    common_path + 'energy_12.pkl',
    common_path + 'energy_13.pkl',
    common_path + 'energy_14.pkl',
    common_path + 'energy_15.pkl',
    common_path + 'energy_16.pkl',
    common_path + 'energy_17.pkl',
]

list_of_x_variable = []
list_of_histories = []
list_of_stats = []
results_summaries = []
for i, results_file in enumerate(results_files):
    with open(results_file, 'rb') as f:
        results_summary = pickle.load(f)
        results_summaries.append(results_summary)

    print(f"Loaded results from: {results_file}")
    detector_config_filename = results_summaries[-1]['config']['detector_file']
    detector = generate_detector(detector_config_filename)
    true_energy = int(np.unique(np.array([e['event_data']['true_energy'] for e in results_summaries[-1]['all_event_results']]))[0])
    list_of_x_variable.append(true_energy)

    all_event_results = results_summary['all_event_results']
    histories = extract_histories(all_event_results)

    # Compute statistics for all metrics
    stats = {}
    for key in histories:
        stats[key] = compute_statistics(histories[key])

    n_events, n_iterations = histories['position_errors'].shape

    # --------------------------
    # Convert energy errors → momentum errors (%)
    # --------------------------
    # Constants
    m_mu = 105.658  # MeV (muon mass)
    T_mu = true_energy   # MeV (kinetic energy)
    E_total = T_mu + m_mu
    p_mu = np.sqrt(E_total**2 - m_mu**2)
    conversion_factor = E_total / p_muG
    momentum_errors_percent = (conversion_factor * (histories['energy_errors'] / E_total)) * 100

    
    histories['momentum_errors_percent'] = momentum_errors_percent
    stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)

    list_of_histories.append(histories)
    list_of_stats.append(stats)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

ci_level_val = 68
line_color = "navy"

def bootstrap_percentile_ci(data, percentile=95, n_bootstrap=1000, ci_level=ci_level_val):
    bootstrap_estimates = []
    n = len(data)
    for _ in range(n_bootstrap):
        bootstrap_sample = np.random.choice(data, size=n, replace=True)
        bootstrap_estimates.append(np.percentile(bootstrap_sample, percentile))
    bootstrap_estimates = np.array(bootstrap_estimates)
    alpha = (100 - ci_level) / 2
    ci_lower = np.percentile(bootstrap_estimates, alpha)
    ci_upper = np.percentile(bootstrap_estimates, 100 - alpha)
    return {
        "estimate": np.percentile(data, percentile),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "std_error": np.std(bootstrap_estimates),
    }

# ==========================================================
# Timing metrics
# ==========================================================
total_times_mean, total_times_std = [], []
adam_times_mean, adam_times_std = [], []

for results_summary in results_summaries:
    all_event_results = results_summary["all_event_results"]
    total_t = np.array([ev["total_event_time"] for ev in all_event_results])
    adam_t = np.array([ev.get("adam_optimization_time", np.nan) for ev in all_event_results])
    total_times_mean.append(np.nanmean(total_t))
    total_times_std.append(np.nanstd(total_t))
    adam_times_mean.append(np.nanmean(adam_t))
    adam_times_std.append(np.nanstd(adam_t))

# ==========================================================
# Bootstrap metrics
# ==========================================================
n_iter = -1
np.random.seed(42)

metrics = {
    "position_errors": {"label": "Pos. error (cm)", "scale": 100},
    "direction_errors": {"label": "Dir. error (°)", "scale": 1},
    "t0_errors": {"label": "t₀ error (ns)", "scale": 1},
    "momentum_errors_percent": {"label": "Mom. error (%)", "scale": 1},
}

valid_histories = [h for h in list_of_histories if isinstance(h.get("position_errors", None), np.ndarray)]
metric_results = {key: {"y": [], "yerr": []} for key in metrics}

for histories in valid_histories:
    for key, meta in metrics.items():
        data = histories[key][:, n_iter] * meta["scale"]
        boot = bootstrap_percentile_ci(data, percentile=68)
        metric_results[key]["y"].append(boot["estimate"])
        yerr = 0.5 * ((boot["ci_upper"] - boot["estimate"]) + (boot["estimate"] - boot["ci_lower"]))
        metric_results[key]["yerr"].append(yerr)

# ==========================================================
# Fit functions
# ==========================================================
def exp_like(x, a, b, c):
    """y = a * b^x + c"""
    return a * (b ** x) + c

def constant(x, q):
    """y = q"""
    return np.full_like(x, q)
    

# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(4, 1, figsize=(6, 8), sharex=True)
plt.subplots_adjust(hspace=0.05)

xdata = np.array(list_of_x_variable)

for ax, (key, meta) in zip(axes[:4], metrics.items()):
    y = np.array(metric_results[key]["y"])
    yerr = np.array(metric_results[key]["yerr"])

    ax.errorbar(
        xdata, y, yerr=yerr, fmt="o", capsize=4, color=line_color, label=meta["label"]
    )

    try:
        if key == "momentum_errors_percent":
            # ---- Constant fit for momentum resolution ----
            popt, _ = curve_fit(
                constant,
                xdata,
                y,
                sigma=yerr,
                absolute_sigma=True,
                p0=[np.mean(y)],
                maxfev=10000,
            )
            xfit = np.linspace(xdata.min(), xdata.max(), 200)
            yfit = constant(xfit, *popt)
            ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)

            q = popt[0]
            ax.text(
                0.55, 0.9,
                f"$y = {q:.2f}$",
                transform=ax.transAxes, fontsize=10, color="cornflowerblue",
                ha="left", va="top",
            )

        else:
            # ---- Exponential-like fit for other metrics ----
            popt, _ = curve_fit(
                exp_like,
                xdata,
                y,
                sigma=yerr,
                absolute_sigma=True,
                p0=[y.max() - y.min(), 0.99, y.min()],
                maxfev=10000,
            )
            xfit = np.linspace(xdata.min(), xdata.max(), 200)
            yfit = exp_like(xfit, *popt)
            ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)

            a, b, c = popt
            ax.text(
                0.55, 0.9,
                f"$y = {a:.2f}\\times\\,{b:.3f}^x + {c:.2f}$",
                transform=ax.transAxes, fontsize=10, color="cornflowerblue",
                ha="left", va="top",
            )

    except RuntimeError:
        print(f"⚠️ Fit failed for {key}")

    ax.set_ylabel(meta["label"])
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

    # Fixed Y-axis limits
    if key == "position_errors":
        ax.set_ylim(10, 35)
    elif key == "direction_errors":
        ax.set_ylim(0., 2.9)
    elif key == "t0_errors":
        ax.set_ylim(0, 1.6)
    elif key == "momentum_errors_percent":
        ax.set_ylim(0, 4.4)

axes[-1].set_xlabel("Energy (MeV)")
fig.align_ylabels(axes)
plt.savefig('figures/tracking_performance_vs_energy.pdf', bbox_inches='tight')
plt.show()